# Notebook 03 — Transformer Forecaster (From Scratch)
**Syllabus Week 8 | CLO 3: Attention Mechanisms — Week 8 Viva Deliverable**

## Problem Statement
Implement a Transformer encoder **from primitives** (no `nn.Transformer`,
`nn.MultiheadAttention`, or `nn.TransformerEncoderLayer`) to forecast the
next 24 hours of `temperature_2m` from a 48-hour window of 7 features.

**This notebook is the Week 8 deliverable.** The viva will ask you to:
1. Explain the Q/K/V projection math.
2. Show the attention heatmap (`attn_sample.npy`).
3. Compare test MSE against the LSTM baseline.

| Hyperparameter | Value |
|---|---|
| Input | (B, 48, 7) |
| Output | (B, 24) |
| Layers | 4 |
| Heads | 4 |
| d_model | 128 |
| FFN dim | 256 |
| Dropout | 0.1 |
| Params | ~400K |

## Section 2 — Math Derivation

### Multi-Head Self-Attention
Given input $X \in \mathbb{R}^{T \times d}$, for head $h$ with $d_k = d/H$:

$$Q_h = X W_h^Q, \quad K_h = X W_h^K, \quad V_h = X W_h^V$$
$$\text{Attention}(Q_h, K_h, V_h) = \text{softmax}\!\left(\frac{Q_h K_h^\top}{\sqrt{d_k}}\right) V_h$$

All heads concatenated and projected:
$$\text{MHSA}(X) = \text{concat}(\text{head}_1, \ldots, \text{head}_H) W^O$$

**Why $\sqrt{d_k}$?** Without scaling, dot products grow with $d_k$,
pushing softmax into saturation regions with near-zero gradients.

### Positional Encoding (sin/cos, no learned embedding)
$$PE_{(\text{pos}, 2i)} = \sin\!\left(\frac{\text{pos}}{10000^{2i/d}}\right), \quad
  PE_{(\text{pos}, 2i+1)} = \cos\!\left(\frac{\text{pos}}{10000^{2i/d}}\right)$$

### Pre-LN Encoder Block
$$x' = x + \text{MHSA}(\text{LayerNorm}(x))$$
$$x'' = x' + \text{FFN}(\text{LayerNorm}(x'))$$

Pre-LN (vs post-LN) stabilises training: gradients flow through the residual
path without passing through LayerNorm, preventing vanishing gradients in deep networks.

### MSE Loss
$$\mathcal{L} = \frac{1}{24}\sum_{h=1}^{24}(\hat{y}_h - y_h)^2$$

In [ ]:
# Section 3 — Data Loading
# TODO: run after python -m ml.data_pipeline
import sys; sys.path.insert(0, '..')
import torch

# from ml.train.train_transformer import load_data
# train_ds, val_ds, test_ds = load_data()
# print(f'train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}')
# X, y = train_ds[0]
# print(f'X: {X.shape}')  # (48, 7)
# print(f'y: {y.shape}')  # (24,)

In [ ]:
# Section 4 — Model Definition & Shape Trace
# TODO: run after deps installed

# from ml.train.train_transformer import WeatherTransformer, MultiHeadSelfAttention, PositionalEncoding
# model = WeatherTransformer()
# print(f'Params: {sum(p.numel() for p in model.parameters()):,}')  # ~400K

# Shape trace through forward pass (B=2, T=48, F=7):
# x_in         : (2, 48, 7)
# input_proj   : (2, 48, 128)      Linear(7 -> 128)
# pos_enc      : (2, 48, 128)      + fixed PE
# block 1-4    : (2, 48, 128)      MHSA + FFN + residuals
# norm         : (2, 48, 128)      final LayerNorm
# mean(dim=1)  : (2, 128)          mean-pool over T=48
# head         : (2, 24)           Linear(128 -> 24)

# Verify with:
# x = torch.randn(2, 48, 7)
# out, attns = model(x, return_attn=True)
# print(f'output: {out.shape}')          # (2, 24)
# print(f'attn[0]: {attns[0].shape}')    # (2, 4, 48, 48)

In [ ]:
# Section 5 — Training Results
# Run: python -m ml.train.train_transformer
# Then paste epoch log below:

# TODO: paste training output here
# epoch 01  train_mse=X.XXXX  val_mse=X.XXXX
# ...
# Early stop / final epoch
# Test MSE=X.XXXX  MAE=X.XXXX

from IPython.display import Image
# Image('../models/transformer/train_curve.png')

## Section 6 — Architecture Diagram

```
Input (B, 48, 7)
      │
 Linear(7 → 128)        ← input projection
      │
 PositionalEncoding      ← sin/cos, no nn.Embedding
      │
  ┌───┴──────────────────────────────────────────┐
  │  TransformerEncoderBlock × 4                 │
  │  ┌──────────────────────────────────────┐    │
  │  │ LayerNorm                            │    │
  │  │ MultiHeadSelfAttention (4 heads)     │    │
  │  │   W_q, W_k, W_v: Linear(128, 128)   │    │
  │  │   scores = Q @ K.T / sqrt(32)       │    │
  │  │   softmax → dropout → @ V → W_o     │    │
  │  │ + residual                           │    │
  │  │ LayerNorm                            │    │
  │  │ FFN: Linear(128→256)→GELU→Linear(256→128) │
  │  │ + residual                           │    │
  │  └──────────────────────────────────────┘    │
  └───────────────────────────────────────────────┘
      │
 LayerNorm
      │
 mean(dim=1)             ← mean-pool T=48 → (B, 128)
      │
 Linear(128 → 24)
      │
 Output (B, 24)          ← next 24h temperature
```

In [ ]:
# Section 7 — Evaluation: Loss Curve + Attention Heatmap
# TODO: run after training
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

# 7a. Training curve
# Image('../models/transformer/train_curve.png')

# 7b. Attention heatmap — the viva visualisation
# attn = np.load('../models/transformer/attn_sample.npy')  # (4, 48, 48)
# fig, axes = plt.subplots(1, 4, figsize=(18, 4))
# for i, ax in enumerate(axes):
#     im = ax.imshow(attn[i], cmap='viridis', aspect='auto')
#     ax.set_title(f'Head {i+1}')
#     ax.set_xlabel('Key (source timestep)')
#     ax.set_ylabel('Query (target timestep)')
#     plt.colorbar(im, ax=ax, fraction=0.046)
# plt.suptitle('WeatherTransformer — Attention Weights (last block, val sample)')
# plt.tight_layout(); plt.show()

# 7c. Forecast vs ground truth (same as notebook 02 for comparison)
# TODO: plot 5 test samples side by side

## Section 8 — Comparison vs LSTM Baseline & Reflection

### Results (fill in after both training runs)
| Model | Params | Test MSE | Test MAE | Train time |
|---|---|---|---|---|
| LSTM (baseline) | ~36K | TODO | TODO | TODO min |
| **Transformer (ours)** | ~400K | TODO | TODO | TODO min |
| Δ improvement | — | TODO% | TODO% | — |

### Reading the attention heatmap
Each head learns to attend to different temporal patterns:
- **Diagonal-heavy heads** → attends to nearby timesteps (local smoothing)
- **Stripe-pattern heads** → attends to the same hour across days (diurnal cycle)
- **Spread-attention heads** → global context, e.g., pressure-temperature coupling

If all heads look identical, training collapsed — check LR and dropout.

### Trade-offs vs LSTM
| Aspect | Transformer | LSTM |
|---|---|---|
| Training speed (CPU) | Faster (parallel) | Slower (sequential) |
| Interpretability | Attention weights | Opaque hidden state |
| Parameter efficiency | Lower (~11× more params) | Higher |
| Long-range dependency | Uniform attention | Fades with distance |

### Constraints honoured
- `nn.Transformer`, `nn.MultiheadAttention`, `nn.TransformerEncoderLayer` — **none used**.
- Q, K, V are explicit `nn.Linear` projections.
- Positional encoding uses the sin/cos formula with `register_buffer`.
- Pre-LN chosen over post-LN for CPU training stability.

### Numbers to fill in
- Best val MSE: **TODO**
- Epoch of early stop: **TODO**
- Attention head with clearest diurnal pattern: **Head TODO**